# Travelling salesman with PySCIPOpt

Minimize the travel cost for visiting `n` customers exactly once. Approach:
- start with an assignment model;
- add cuts until there are no sub-cycles, with two cutting-plane options (used inside `solve_tsp`):
    - `addcut`: limit the number of edges in a connected component `S` to `|S| - 1`;
    - `addcut2`: require the number of edges between two connected components to be `>= 2`.

Run the cells below, then **click on the canvas to add nodes** and use the **Solve** / **Reset** buttons.

Example derived from the [PySCIPOpt repository](https://github.com/scipopt/PySCIPOpt) under MIT license.
Copyright (c) by Joao Pedro PEDROSO and Mikio KUBO, 2012

In [ ]:
import math
import random

import networkx
import pyscipopt as scip

import tsp_cuts_utils as utils


def solve_tsp(
    V: list[int], c: dict[tuple[int, int], float]
) -> tuple[float, list[tuple[int, int]]]:
    """solve_tsp -- solve the traveling salesman problem
       - start with assignment model
       - add cuts until there are no sub-cycles
    Parameters:
        - V: set/list of nodes in the graph
        - c[i,j]: cost for traversing edge (i,j)
    Returns the optimum objective value and the list of edges used.
    """
    if len(V) < 2:  # nothing to route: avoid an infinite cut loop on empty input
        return 0.0, []

    def addcut(cut_edges: list[tuple[int, int]]) -> bool:
        G = networkx.Graph()
        G.add_edges_from(cut_edges)
        Components = list(networkx.connected_components(G))
        if len(Components) == 1:
            return False
        model.freeTransform()
        for S in Components:
            model.addCons(scip.quicksum(x[i, j] for i in S for j in S if j > i) <= len(S) - 1)
            print("cut: len(%s) <= %s" % (S, len(S) - 1))
        return True

    def addcut2(cut_edges: list[tuple[int, int]]) -> bool:
        G = networkx.Graph()
        G.add_edges_from(cut_edges)
        Components = list(networkx.connected_components(G))

        if len(Components) == 1:
            return False
        model.freeTransform()
        for S in Components:
            T = set(V) - set(S)
            print("S:", S)
            print("T:", T)
            model.addCons(scip.quicksum(x[i, j] for i in S for j in T if j > i) +
                          scip.quicksum(x[i, j] for i in T for j in S if j > i) >= 2)
            print("cut: %s >= 2" % "+".join([("x[%s,%s]" % (i, j)) for i in S for j in T if j > i]))
        return True

    # main part of the solution process:
    model = scip.Model("tsp")

    model.hideOutput()  # silent/verbose mode
    x = {}
    for i in V:
        for j in V:
            if j > i:
                x[i, j] = model.addVar(ub=1, name="x(%s,%s)" % (i, j))

    for i in V:
        model.addCons(scip.quicksum(x[j, i] for j in V if j < i) + \
                      scip.quicksum(x[i, j] for j in V if j > i) == 2, "Degree(%s)" % i)

    model.setObjective(scip.quicksum(c[i, j] * x[i, j] for i in V for j in V if j > i), "minimize")

    EPS = 1.e-6
    isMIP = False
    while True:
        model.optimize()
        edges = []
        for (i, j) in x:
            if model.getVal(x[i, j]) > EPS:
                edges.append((i, j))

        if addcut(edges) == False:
            if isMIP:  # integer variables, components connected: solution found
                break
            model.freeTransform()
            for (i, j) in x:  # all components connected, switch to integer model
                model.chgVarType(x[i, j], "B")
                isMIP = True

    return model.getObjVal(), edges


def distance(x1: float, y1: float, x2: float, y2: float) -> float:
    """distance: euclidean distance between (x1,y1) and (x2,y2)"""
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

In [ ]:
utils.tsp_widget(solve_tsp)